In [1]:
import os

from plistsync.logger import log

# Update you config directory as needed
os.environ["PSYNC_CONFIG_DIR"] = "../config"


/mnt/Data/micromamba/envs/plistsync/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Sync example Tidal and Spotify

In this example we will show how to sync a playlist from Spotify to Tidal. This is a pretty easy case as tracks
on both services can be identified by their [ISRC](https://en.wikipedia.org/wiki/International_Standard_Recording_Code).

To execute this example you will need to configure both the spotify and tidal services.


## The initial playlist

We start with a spotify playlist here and create a tidal playlist with the same name if it does not exist yet. You can
also start with two blank playlists.


In [3]:
from plistsync.services.spotify import SpotifyPlaylistCollection

# Spotify id of the playlist to sync (Drum & Bass Top 100)
source_spotify_id = "0Zarq4BVkFkZOWkmqsfrjA"
spotify_playlist = await SpotifyPlaylistCollection.from_id(source_spotify_id)

In [4]:
spotify_playlist.name

'Drum and Bass Top 100'

Next up we create a corresponding playlist on Tidal (or get it if it already exists):

In [5]:
from plistsync.services.tidal import TidalLibraryCollection

tidal_library = TidalLibraryCollection()

if tidal_library.has_playlist(spotify_playlist.name):
    # Playlist exists, we can use it
    _tidal_playlists = tidal_library.get_playlist(spotify_playlist.name)
    assert _tidal_playlists is not None
    tidal_playlists = _tidal_playlists
    log.info(f"Using existing Tidal playlist '{tidal_playlists.id}'")
else:
    # Create the playlist
    tidal_playlists = tidal_library.create_playlist(spotify_playlist.name, description="Imported from Spotify")
    log.info(f"Created Tidal playlist '{tidal_playlists.id}'")


INFO:plistsync:Using existing Tidal playlist '5e9f3d7e-325d-4cbf-afde-9538b0aa8819'


In [6]:
tidal_playlists.name

'Drum and Bass Top 100'

We now have both playlists and can start to sync them. We will add tracks that are in the Spotify playlist but not in the Tidal one.

In [7]:
tracks_spotify = list(spotify_playlist)
tracks_tidal = list(tidal_playlists)

The following might not be super efficient, but it works for demonstration purposes. Maybe we can optimize it later?

In [8]:
from plistsync.services.spotify import SpotifyPlaylistTrack

# Find tracks in Spotify playlist that are not in Tidal playlist
missing_tracks: list[SpotifyPlaylistTrack] = []

for track_s in spotify_playlist:
    # Try to match by ISRC only (cutoff 1.0)
    matches = tidal_playlists.match(track_s, cutoff=1.0)
    if len(matches.found) == 0:
        missing_tracks.append(track_s)

log.info(f"Found {len(missing_tracks)} missing tracks in Tidal playlist '{tidal_playlists.name}'")


tidal_tracks = tidal_library.find_many_by_global_ids(map(lambda t: t.global_ids, missing_tracks))
tidal_tracks = list(filter(None,tidal_tracks))
log.info(f"Found {len(tidal_tracks)} out of {len(missing_tracks)} tracks on Tidal")


tidal_playlists.add_tracks(tidal_tracks)
log.info(f"Tidal playlist '{tidal_playlists.name}' now has {len(tidal_playlists)} tracks")

INFO:plistsync:Found 30 missing tracks in Tidal playlist 'Drum and Bass Top 100'
INFO:plistsync:Found 30 out of 30 tracks on Tidal
INFO:plistsync:Added 30 tracks to playlist 'Drum and Bass Top 100'
INFO:plistsync:Tidal playlist 'Drum and Bass Top 100' now has 130 tracks


One directional sync is now done. More features coming soon!